# MVB-02 — Data Preparation & Validation

**Project:** P001-MVB — Explainable and Responsible AI for Differentiating Bacterial and Viral Meningitis: A Clinical Decision Support System
**Stage:** 02_MVB_Data-Preparation
**Input:** MVB-01-dataset-raw/meningitis.csv
**Date:** 22 July 2026

This notebook validates the raw dataset against clinical plausibility and resolves the open data-quality questions identified in Stage 01, before producing a cleaned dataset for exploratory data analysis.

## Outputs

This stage produces:
- Cleaned dataset (`MVB-02-meningitis-cleaned.csv`)
- Preprocessing summary log (`MVB-02-preprocessing-summary.csv`)
- Data preparation notebook
- Updated data preparation documentation

In [ ]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Set directory
import os
os.chdir('/content/drive/MyDrive/P001-MVB/02_MVB_Data-Preparation')

In [ ]:
# Imports
import pandas as pd

In [ ]:
# Load from Stage 01's raw data folder (input to this stage)
df = pd.read_csv("../01_MVB_Data-Acquisition/MVB-01-dataset-raw/meningitis.csv")
print("Shape loaded:", df.shape)

Shape loaded: (1200, 14)


In [ ]:

# Confirm the data matches what Stage 01 documented — no silent changes between stages
assert df.shape == (1200, 14)
assert df["Patient_ID"].duplicated().sum() == 0
print("Integrity check passed: shape and Patient_ID uniqueness match Stage 01 findings.")

Integrity check passed: shape and Patient_ID uniqueness match Stage 01 findings.


## 1. Target Variable Resolution — Handling "Unknown" Diagnosis

The decision to exclude the "Unknown" diagnosis class is based primarily on the project's binary classification objective (Bacterial vs. Viral). No clinically defensible rule exists to reassign these 67 records to either class, so exclusion is the only consistent option.

The additional analyses in this stage (Age>100 overlap, lab profile comparison) provide further context on the characteristics of these records, but were not the primary basis for exclusion.

In [ ]:
print("Diagnosis distribution before filtering:")
print(df["Diagnosis"].value_counts())

# Drop Unknown diagnosis rows — target variable must be binary
df_clean = df[df["Diagnosis"] != "Unknown"].copy()

print()
print("Diagnosis distribution after filtering:")
print(df_clean["Diagnosis"].value_counts())
print()
print("Rows removed:", len(df) - len(df_clean))

Diagnosis distribution before filtering:
Diagnosis
Bacterial    595
Viral        538
Unknown       67
Name: count, dtype: int64

Diagnosis distribution after filtering:
Diagnosis
Bacterial    595
Viral        538
Name: count, dtype: int64

Rows removed: 67


## 2. Age Plausibility Check

43 rows have Age > 100. Investigating whether these represent valid extreme ages or records with greater uncertainty.

In [ ]:
age_gt100 = df[df["Age"] > 100]
print("Age>100 row count:", len(age_gt100))
print()
print("Diagnosis breakdown within Age>100 group:")
print(age_gt100["Diagnosis"].value_counts())
print()
print("Unknown share within Age>100 group: {:.1f}%".format(
    (age_gt100["Diagnosis"] == "Unknown").mean() * 100))
print("Unknown share in full dataset: {:.1f}%".format(
    (df["Diagnosis"] == "Unknown").mean() * 100))

Age>100 row count: 43

Diagnosis breakdown within Age>100 group:
Diagnosis
Unknown      17
Bacterial    14
Viral        12
Name: count, dtype: int64

Unknown share within Age>100 group: 39.5%
Unknown share in full dataset: 5.6%


In [ ]:
# Compare CSF/blood markers: Age>100 group vs rest of dataset
cols = ["WBC_Count", "Protein_Level", "Glucose_Level", "Hemoglobin",
        "WBC_Blood_Count", "Platelets", "CRP_Level"]

comparison = pd.DataFrame({
    "Age>100 mean": age_gt100[cols].mean(),
    "Rest mean": df[df["Age"] <= 100][cols].mean()
}).round(1)

print(comparison)

                 Age>100 mean  Rest mean
WBC_Count             15079.0    11722.3
Protein_Level           165.9       98.6
Glucose_Level            67.8       52.0
Hemoglobin                9.3        9.7
WBC_Blood_Count       12251.5    11700.7
Platelets            179668.7   197488.0
CRP_Level                47.2       23.7


**Decision:** Age>100 rows (n=43) are retained — values are complete and non-placeholder, so there is no clear justification for removal. However, this subgroup shows a disproportionate share of "Unknown" diagnoses (39.5% vs. 5.6% baseline) and a lab profile that does not cleanly track either diagnosis pattern, suggestive of greater uncertainty within this subgroup. This is documented as a limitation rather than treated as grounds for exclusion.

## 3. Numeric Range Validation

Comparing dataset ranges against typical real-world clinical reference ranges for each lab marker.

In [ ]:

lab_cols = ["WBC_Count", "Protein_Level", "Glucose_Level", "Hemoglobin",
            "WBC_Blood_Count", "Platelets", "CRP_Level"]

for col in lab_cols:
    print(f"{col}: min={df[col].min()}, max={df[col].max()}, mean={df[col].mean():.1f}")

print()
neg_found = False
for col in lab_cols:
    neg = (df[col] < 0).sum()
    if neg > 0:
        print(f"{col} has {neg} negative values")
        neg_found = True
if not neg_found:
    print("No negative values found across any lab marker.")

WBC_Count: min=2008, max=24717, mean=11842.6
Protein_Level: min=2, max=299, mean=101.0
Glucose_Level: min=0, max=148, mean=52.6
Hemoglobin: min=1, max=18, mean=9.7
WBC_Blood_Count: min=4022, max=19991, mean=11720.4
Platelets: min=100088, max=399479, mean=196849.5
CRP_Level: min=0, max=99, mean=24.6

No negative values found across any lab marker.


**Finding:** All values are non-negative and internally consistent. Most markers (Protein, Glucose, Hemoglobin, Platelets, CRP) fall within plausible real-world clinical ranges.

`WBC_Count` (CSF) is the exception — the dataset range (2,008–24,717, mean 11,842) is far higher than real-world CSF white cell counts, where normal is <5 and even severe bacterial meningitis rarely exceeds 10,000. This suggests that the dataset may use measurement units, scaling conventions, or preprocessing approaches that differ from standard clinical reporting. Because the accompanying documentation does not specify these details, the exact explanation cannot be confirmed.

This does not invalidate the project — the *relative* separation between Bacterial and Viral classes (confirmed in Stage 01) still holds and is what the model learns from. It does mean the dataset should be described as **clinically-inspired but not clinically-calibrated** in the Model Card and Limitations sections, rather than implying real-world reference ranges apply directly.

## 4. Final Feature Set Confirmation

In [ ]:
print("Cleaned dataset shape:", df_clean.shape)
print()
print("Columns retained for modeling:")
print(df_clean.columns.tolist())

Cleaned dataset shape: (1133, 14)

Columns retained for modeling:
['Patient_ID', 'Age', 'Gender', 'WBC_Count', 'Protein_Level', 'Glucose_Level', 'Pathogen_Present', 'Diagnosis', 'Outcome', 'Hemoglobin', 'WBC_Blood_Count', 'Platelets', 'CRP_Level', 'Risk_Level']


## 5. Save Cleaned Dataset

In [ ]:

df_clean.to_csv("MVB-02-cleaned-data/MVB-02-meningitis-cleaned.csv", index=False)
print("Saved cleaned dataset:", df_clean.shape)
print("Dataset version: MVB-02-meningitis-cleaned.csv")

Saved cleaned dataset: (1133, 14)
Dataset version: MVB-02-meningitis-cleaned.csv


In [ ]:
preprocessing_summary = pd.DataFrame({
    "Metric": [
        "Original rows",
        "Rows removed",
        "Final rows",
        "Final columns"
    ],
    "Value": [
        len(df),
        len(df) - len(df_clean),
        len(df_clean),
        df_clean.shape[1]
    ]
})

preprocessing_summary.to_csv(
    "MVB-02-cleaned-data/MVB-02-preprocessing-summary.csv",
    index=False
)

print(preprocessing_summary)

          Metric  Value
0  Original rows   1200
1   Rows removed     67
2     Final rows   1133
3  Final columns     14


## 6. Summary

- **Integrity check:** confirmed dataset shape and Patient_ID uniqueness match Stage 01 findings before preprocessing began.
- **Unknown diagnosis (n=67):** dropped — primary justification is the project's binary classification objective; supporting context (Age>100 overlap) noted separately, not treated as a co-justification.
- **Age>100 (n=43):** retained, documented as a limitation — values are complete, but the subgroup shows signals of greater uncertainty (disproportionate Unknown overlap, inconsistent lab profile).
- **Lab ranges:** all non-negative and internally consistent. `WBC_Count` scale does not match real-world clinical values — exact cause unconfirmed; dataset is described as clinically-inspired but not clinically-calibrated, to be noted explicitly in the Model Card.
- **Cleaned dataset:** saved to `MVB-02-cleaned-data/MVB-02-meningitis-cleaned.csv`, ready for Stage 03 (EDA).

## Deliverables Produced

✓ Cleaned dataset (MVB-02-meningitis-cleaned.csv)
✓ Preprocessing summary log (MVB-02-preprocessing-summary.csv)
✓ Data preparation notebook
✓ Updated documentation

**Next Stage:** 03_MVB_EDA